# Energy & Development — Irrigation Energy Calculator

In [9]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import warnings
import os
import random
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore')

# ── Edit this path ─────────────────────────────────────────────────────────
DATA_DIR = "/Users/mackenziee/Documents/YSE/Semester 4/Energy and Development/Final_Project/crop_data/"

# ══════════════════════════════════════════════════════════════════════════
# PRICES  (USD, April 2026) — edit as needed
# ══════════════════════════════════════════════════════════════════════════
diesel_price_per_l        = 1.74    # USD / litre
electricity_price_per_kwh = 0.16    # USD / kWh  (grid tariff)

# ══════════════════════════════════════════════════════════════════════════
# CAPITAL EXPENDITURE  (USD one-time)
# ══════════════════════════════════════════════════════════════════════════
CAPEX = {
    "diesel":    700,   # 5 HP diesel pump + install
    "grid_elec": 300,   # 0.5 kW electric pump + wiring
    "solar_s":   550,   # 0.12 kW solar kit  (cedarsolar Spitfire-50 + panel)
    "solar_l":   600,   # 0.5 kW solar kit   (cedarsolar Spitfire-100)
}

# Annual maintenance / servicing (USD / yr)
MAINTENANCE_YR = {
    "diesel":    60,   # oil, filters, service
    "grid_elec":  15,
    "solar_s":    15,   # occasional cleaning / fuse
    "solar_l":    15,
}

TCO_YEARS = 5           # cost-of-ownership horizon

# ══════════════════════════════════════════════════════════════════════════
# EMISSIONS FACTORS
# ══════════════════════════════════════════════════════════════════════════
diesel_ghg_kgCO2_per_kwh = 0.255   # kg CO2 / kWh diesel energy output (fixed)
# Grid electricity: loaded per-country from carbon-intensity-electricity.csv
# Solar: 0 operational GHG

# ══════════════════════════════════════════════════════════════════════════
# PUMP / EMITTER PHYSICS
# ══════════════════════════════════════════════════════════════════════════

# Emitter hydraulic parameters (independent of pump type)
EMITTER_PARAMS = {
    "drip": {
        "pumping_head":                  15 * 0.703,   # psi → m head
        "field_pipe_head_mult_per_acre":  6 * 0.703,
        "water_effic": 0.93,
        "pipe_to_field": 0.06,           # head loss per m of supply pipe (m/m)
        "head_for_fric_loss": 5,         # m, well-to-surface friction
        "filter_and_injector": 15 * 0.703,
    },
    "sprinkler": {
        "pumping_head":                  15 * 0.703,
        "field_pipe_head_mult_per_acre":  0,
        "water_effic": 0.70,
        "pipe_to_field": 0.06,
        "head_for_fric_loss": 5,
        "filter_and_injector": 15 * 0.703,
    },
    "furrough/ canal": {
        "pumping_head":                  15 * 0.703,
        "field_pipe_head_mult_per_acre":  0,
        "water_effic": 0.65,
        "pipe_to_field": 0.06,
        "head_for_fric_loss": 5,
        "filter_and_injector": 15 * 0.703,
    },
}

# Flow rates in m³/s, split by pipe category then emitter type.
# Diesel pumps use 2" outlet pipes → higher flow rates.
# Electric/solar pumps use ≤1" outlet pipes → lower flow rates.
FLOW_RATES = {
    "diesel": {          # 2" outlet pipe
        "drip":            12 * 6.309e-5,   # 12 GPM
        "sprinkler":       20 * 6.309e-5,   # 20 GPM
        "furrough/ canal": 25 * 6.309e-5,   # 25 GPM
    },
    "electric": {        # 1" outlet pipe
        "drip":             5 * 6.309e-5,   #  5 GPM
        "sprinkler":       10 * 6.309e-5,   # 10 GPM
        "furrough/ canal":  8 * 6.309e-5,   #  8 GPM
    },
}

# Map each fuel type to a pipe category
FUEL_PIPE_CAT = {
    "diesel":    "diesel",
    "grid_elec": "electric",
    "solar_s":   "electric",
    "solar_l":   "electric",
}

# Pump-to-shaft efficiencies
FUEL_EFFICIENCIES = {
    "diesel":    0.30,
    "grid_elec": 0.75,
    "solar_s":   0.75,
    "solar_l":   0.75,
}

# Maximum continuous power output per pump type (kW)
MAX_POWER_KW = {
    "diesel":    3.677,   # 5 HP
    "grid_elec": 0.500,
    "solar_s":   0.120,
    "solar_l":   0.500,
}

FUEL_LABELS = {
    "diesel":    "Diesel",
    "grid_elec": "Grid Electric",
    "solar_s":   "Solar (small)",
    "solar_l":   "Solar (large)",
}
BAR_COLORS = {
    "diesel":    "#e07b54",
    "grid_elec": "#4c72b0",
    "solar_s":   "#f6c244",
    "solar_l":   "#f6a800",
}


# ══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════

def calc_water_demand(emitter_type, fuel_type, pump_params):
    """Return hydraulic dict for a given emitter + fuel (pipe-size) combination."""
    ep        = EMITTER_PARAMS[emitter_type]
    flow_rate = FLOW_RATES[FUEL_PIPE_CAT[fuel_type]][emitter_type]

    static_head      = (pump_params["pump_depth"] + pump_params["discharge_height"]
                        + ep["head_for_fric_loss"])
    emitter_pressure = (ep["pumping_head"]
                        + ep["field_pipe_head_mult_per_acre"] * pump_params["field_size"])
    TDH = (static_head + emitter_pressure + ep["filter_and_injector"]
           + ep["pipe_to_field"] * pump_params["distance_to_discharge"])

    mm_per_sec = (flow_rate / (pump_params["field_size"] * 4046.86)) * 1000 * ep["water_effic"]
    return {"mm_water_per_sec": mm_per_sec, "flow_rate": flow_rate, "TDH": TDH}


def calc_power_kw(water_demand, fuel_type):
    eff = FUEL_EFFICIENCIES[fuel_type]
    return (9.81 * water_demand["flow_rate"] * water_demand["TDH"]) / eff


def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


print("Config loaded — proceed to Step 1 below.")


Config loaded — proceed to Step 1 below.


## Step 1 — Select Crop

In [14]:
# ── Shared state dict ──────────────────────────────────────────────────────
S = {}

excel_file_path = os.path.join(DATA_DIR, "Data_Note.xlsx")
crop_names_dict = {}
if os.path.exists(excel_file_path):
    try:
        df_crops = pd.read_excel(excel_file_path)
        if 'cap_crop_name' in df_crops.columns:
            crop_series = df_crops['cap_crop_name'].dropna().drop_duplicates()
            crop_names_dict = {n: i for i, n in enumerate(crop_series.tolist())}
        else:
            print("Column 'cap_crop_name' not found in Data_Note.xlsx")
    except Exception as e:
        print(f"Error reading crop names: {e}")
else:
    print(f"File not found: {excel_file_path}")

_desc = {'description_width': 'initial'}
_w320 = widgets.Layout(width='340px')
_w460 = widgets.Layout(width='480px')
_ds   = {'description_width': '260px'}

# ── Step 1 ─────────────────────────────────────────────────────────────────
crop_selector = widgets.Dropdown(options=list(crop_names_dict.keys()),
                                 description='Crop:', style=_desc, layout=_w320)
btn_load_crop = widgets.Button(description='Load Crop Data', button_style='primary',
                               layout=widgets.Layout(width='160px'))
out_crop = widgets.Output()

# ── Step 2 ─────────────────────────────────────────────────────────────────
country_selector   = widgets.Dropdown(options=[], description='Country:', style=_desc, layout=_w320)
btn_select_country = widgets.Button(description='Confirm Country', button_style='primary',
                                    layout=widgets.Layout(width='170px'))
out_country = widgets.Output()
step2_box = widgets.VBox([widgets.HTML('<h3>Step 2 — Select Country</h3>'),
                          country_selector, btn_select_country, out_country],
                         layout=widgets.Layout(display='none'))

# ── Step 3 ─────────────────────────────────────────────────────────────────
region_selector   = widgets.Dropdown(options=[], description='Region:', style=_desc, layout=_w320)
btn_select_region = widgets.Button(description='Confirm Region', button_style='primary',
                                   layout=widgets.Layout(width='170px'))
out_region = widgets.Output()
step3_box = widgets.VBox([widgets.HTML('<h3>Step 3 — Select Region</h3>'),
                          region_selector, btn_select_region, out_region],
                         layout=widgets.Layout(display='none'))

# ── Step 4 ─────────────────────────────────────────────────────────────────
w_depth     = widgets.BoundedIntText(value=10,  min=1,  max=500,   description='Well depth (m):', style=_ds, layout=_w460)
w_discharge = widgets.BoundedIntText(value=2,   min=0,  max=200,   description='Discharge height above ground (m):', style=_ds, layout=_w460)
w_distance  = widgets.BoundedIntText(value=50,  min=1,  max=5000,  description='Well-to-field distance (m):', style=_ds, layout=_w460)
w_field     = widgets.BoundedIntText(value=5,   min=1,  max=10000, description='Field size (acres):', style=_ds, layout=_w460)
w_emitter   = widgets.Dropdown(options=list(EMITTER_PARAMS.keys()),
                                description='Emitter type:', style=_ds, layout=_w460)
btn_set_pump = widgets.Button(description='Confirm Pump Params', button_style='primary',
                               layout=widgets.Layout(width='200px'))
out_pump = widgets.Output()
step4_box = widgets.VBox([widgets.HTML('<h3>Step 4 — Pump Parameters</h3>'),
                          w_depth, w_discharge, w_distance, w_field, w_emitter,
                          btn_set_pump, out_pump],
                         layout=widgets.Layout(display='none'))

# ── Step 5 ─────────────────────────────────────────────────────────────────
w_lat = widgets.FloatText(value=0.0, description='Farm latitude  (0,0 = random):', style=_ds, layout=_w460)
w_lon = widgets.FloatText(value=0.0, description='Farm longitude (0,0 = random):', style=_ds, layout=_w460)
btn_run     = widgets.Button(description='Run Analysis', button_style='success',
                              layout=widgets.Layout(width='160px'))
out_results = widgets.Output()
step5_box = widgets.VBox([widgets.HTML('<h3>Step 5 — Farm Location &amp; Results</h3>'),
                          widgets.HTML('<p>Enter decimal-degree coordinates. '
                                       '<b>Leave both at 0 to randomly sample a point within the region.</b></p>'),
                          w_lat, w_lon, btn_run, out_results],
                         layout=widgets.Layout(display='none'))


# ══════════════════════════════════════════════════════════════════════════
# CALLBACKS
# ══════════════════════════════════════════════════════════════════════════

def on_load_crop(b):
    with out_crop:
        clear_output()
        S['selected_crop'] = crop_selector.value
        term = S['selected_crop'].replace(' ', '_').lower()
        found = None
        try:
            for fn in os.listdir(DATA_DIR):
                if fn.lower().endswith('.csv') and term in fn.lower():
                    found = fn; break
        except FileNotFoundError:
            print(f"Directory not found: {DATA_DIR}"); return
        if not found:
            print(f"No CSV found for '{S['selected_crop']}'"); return
        try:
            S['crop_df'] = pd.read_csv(os.path.join(DATA_DIR, found))
            print(f"Loaded {found}  ({len(S['crop_df'])} rows)")
        except Exception as e:
            print(f"Error: {e}"); return
        country_selector.options = sorted(S['crop_df']['ADM0_NAME'].dropna().unique().tolist())
        step2_box.layout.display = ''


def on_select_country(b):
    with out_country:
        clear_output()
        country = country_selector.value
        S['selected_country'] = country
        filtered = S['crop_df'][S['crop_df']['ADM0_NAME'] == country].copy()

        grid_path  = os.path.join(DATA_DIR, "GlobalGridID_2020.xlsx")
        cache_path = os.path.join(DATA_DIR,
            f"merged_{S['selected_crop'].replace(' ','_')}_{country.replace(' ','_')}.parquet")

        if os.path.exists(cache_path):
            try:
                S['merged_df'] = pd.read_parquet(cache_path)
                print(f"Loaded cached merge  {S['merged_df'].shape}")
            except Exception as e:
                print(f"Cache failed ({e}) — re-merging...")
                S['merged_df'] = _do_merge(filtered, grid_path, country, cache_path)
        else:
            S['merged_df'] = _do_merge(filtered, grid_path, country, cache_path)
        if S.get('merged_df') is None:
            return

        # Load carbon intensity as a scalar
        S['carbon_gCO2_per_kwh'] = 0.0
        ei_path = os.path.join(DATA_DIR, "carbon-intensity-electricity.csv")
        if os.path.exists(ei_path):
            try:
                ei = pd.read_csv(ei_path)
                ei_c = ei[ei['Entity'] == country]
                if not ei_c.empty:
                    ei_l = ei_c[ei_c['Year'] == ei_c['Year'].max()]
                    ci_col = next((c for c in ei_l.columns
                                   if any(k in c.lower() for k in ['carbon','intensity','gco2','co2'])
                                   and c not in ('Entity','Code','Year')), None)
                    if ci_col:
                        S['carbon_gCO2_per_kwh'] = float(ei_l[ci_col].iloc[0])
                        print(f"Grid carbon intensity: {S['carbon_gCO2_per_kwh']:.1f} gCO2/kWh")
                    else:
                        print("Carbon intensity column not identified — grid GHG set to 0")
                else:
                    print(f"'{country}' not in emissions CSV — grid GHG set to 0")
            except Exception as e:
                print(f"Error reading emissions CSV: {e}")
        else:
            print("carbon-intensity-electricity.csv not found")

        region_selector.options = sorted(S['merged_df']['ADM1_NAME'].dropna().unique().tolist())
        step3_box.layout.display = ''


def _do_merge(filtered, grid_path, country, cache_path):
    if not os.path.exists(grid_path):
        print(f"Grid file not found: {grid_path}"); return None
    try:
        print("Reading GlobalGridID_2020.xlsx — this may take a moment...")
        grid_df = pd.read_excel(grid_path)
        gf = grid_df[grid_df['ADM0_NAME'] == country].copy()
        merged = pd.merge(filtered, gf, left_on='grid_code', right_on='SPAM_GridCode', how='inner')
        print(f"Merged — shape: {merged.shape}")
        merged.to_parquet(cache_path, index=False)
        print(f"Cached to {os.path.basename(cache_path)}")
        return merged
    except Exception as e:
        print(f"Merge error: {e}"); return None


def on_select_region(b):
    with out_region:
        clear_output()
        S['selected_region'] = region_selector.value
        crop_df = S['merged_df'][S['merged_df']['ADM1_NAME'] == S['selected_region']].copy()
        remap = {}
        for col in crop_df.columns[6:]:
            try:    remap[col] = col.split('_', 2)[2]
            except: remap[col] = col
        crop_df.rename(columns=remap, inplace=True)
        S['crop_df_region'] = crop_df
        print(f"Region set to {S['selected_region']}  ({len(crop_df)} grid points)")
        step4_box.layout.display = ''


def on_set_pump(b):
    with out_pump:
        clear_output()
        S['pump_params'] = {
            "pump_depth":            w_depth.value,
            "discharge_height":      w_discharge.value,
            "distance_to_discharge": w_distance.value,
            "field_size":            w_field.value,
            "emitter_type":          w_emitter.value,
        }
        print("Pump parameters saved:")
        for k, v in S['pump_params'].items():
            print(f"  {k}: {v}")
        step5_box.layout.display = ''


def on_run(b):
    with out_results:
        clear_output()
        pump    = S['pump_params']
        ci      = S.get('carbon_gCO2_per_kwh', 0.0)
        emitter = pump['emitter_type']

        # ── 1. Build coordinate dataframe ──────────────────────────────────
        map_df = S['crop_df_region'].copy()
        map_df['Y_CORD'] = pd.to_numeric(map_df['Y_CORD'], errors='coerce')
        map_df['X_CORD'] = pd.to_numeric(map_df['X_CORD'], errors='coerce')
        map_df = map_df.dropna(subset=['Y_CORD', 'X_CORD']).reset_index(drop=True)
        if map_df.empty:
            print("No valid coordinates."); return

        # ── 2. Farm location ────────────────────────────────────────────────
        if w_lat.value == 0.0 and w_lon.value == 0.0:
            row0 = map_df.sample(1, random_state=random.randint(0, 9999)).iloc[0]
            user_lat, user_lon = float(row0['Y_CORD']), float(row0['X_CORD'])
            print(f"Randomly selected point within {S['selected_region']}: "
                  f"({user_lat:.4f}, {user_lon:.4f})")
        else:
            user_lat, user_lon = float(w_lat.value), float(w_lon.value)
            print(f"Using entered coordinates: ({user_lat:.4f}, {user_lon:.4f})")

        # ── 3. Closest grid point ───────────────────────────────────────────
        dists   = [haversine(user_lat, user_lon, r['Y_CORD'], r['X_CORD'])
                   for _, r in map_df.iterrows()]
        closest = map_df.iloc[np.argmin(dists)]

        # ── 4. Map ──────────────────────────────────────────────────────────
        center = [map_df['Y_CORD'].mean(), map_df['X_CORD'].mean()]
        m = folium.Map(location=center, zoom_start=8)
        for _, row in map_df.iterrows():
            folium.CircleMarker([row['Y_CORD'], row['X_CORD']], radius=4,
                                color='steelblue', fill=True, fill_opacity=0.6,
                                popup=f"Grid: {row['grid_code']}").add_to(m)
        folium.Marker([user_lat, user_lon], popup='Your farm',
                      icon=folium.Icon(color='red', icon='home')).add_to(m)
        folium.Marker([closest['Y_CORD'], closest['X_CORD']],
                      popup=f"Closest: {closest['grid_code']}",
                      icon=folium.Icon(color='green', icon='star')).add_to(m)
        display(m)
        print(f"Closest grid: {closest['grid_code']} at "
              f"({closest['Y_CORD']:.3f}, {closest['X_CORD']:.3f}) — {min(dists):.1f} km")

        # ── 5. Monthly water data ───────────────────────────────────────────
        bl_cols = [c for c in closest.index if c.startswith('bl_') and c != 'bl_ann']
        gn_cols = [c for c in closest.index if c.startswith('gn_rf') and c != 'gn_rf_ann']
        monthly_bl = pd.Series([closest[c] for c in bl_cols],
                                index=[c.replace('bl_', '') for c in bl_cols])
        monthly_gn = pd.Series([closest[c] for c in gn_cols],
                                index=[c.replace('gn_rf_', '') for c in gn_cols])
        monthly_water_use = monthly_bl * pump["field_size"]*  4046.86            #convert acres to L

        # ── 6. Per-fuel physics (flow rate varies by fuel type) ─────────────
        months_order = ['Jan','Feb','Mar','Apr','May','Jun',
                        'Jul','Aug','Sep','Oct','Nov','Dec']
        usage = pd.DataFrame({'water_use_mm': monthly_bl, 'rainfall_mm': monthly_gn, "water_use_volume" : monthly_water_use})

        fuel_water = {}
        power_errors = []
        for ft in FUEL_EFFICIENCIES:
            wd  = calc_water_demand(emitter, ft, pump)
            pwr = calc_power_kw(wd, ft)
            pump_min = (monthly_bl / wd['mm_water_per_sec']) / 60
            pump_hr  = pump_min / 60
            energy   = (pwr * pump_min) / 60

            usage[f'pump_hr_{ft}']     = pump_hr
            usage[f'power_{ft}_kW']    = pwr
            usage[f'energy_{ft}_kWh']  = energy
            fuel_water[ft] = wd

            if pwr > MAX_POWER_KW[ft]:
                power_errors.append(
                    f"{FUEL_LABELS[ft]}: required {pwr:.2f} kW > capacity {MAX_POWER_KW[ft]:.2f} kW")

        # ── 7. OPEX columns ─────────────────────────────────────────────────
        usage['opex_diesel_usd']    = usage['energy_diesel_kWh'] * diesel_price_per_l / 10.5
        usage['opex_grid_elec_usd'] = usage['energy_grid_elec_kWh'] * electricity_price_per_kwh
        usage['opex_solar_s_usd']   = MAINTENANCE_YR['solar_s'] / 12   # maintenance only
        usage['opex_solar_l_usd']   = MAINTENANCE_YR['solar_l'] / 12

        # ── 8. GHG ──────────────────────────────────────────────────────────
        usage['ghg_diesel_kg']    = usage['energy_diesel_kWh'] * diesel_ghg_kgCO2_per_kwh
        usage['ghg_grid_elec_kg'] = usage['energy_grid_elec_kWh'] * (ci / 1000.0)
        usage['ghg_solar_s_kg']   = 0.0
        usage['ghg_solar_l_kg']   = 0.0

        S['usage'] = usage

        # ── 9. Line chart: water & pumping time (diesel) ────────────────────
        usage_s = usage.reindex(months_order)
        fig1, ax1 = plt.subplots(figsize=(12, 5))
        ax1.plot(usage_s.index, usage_s['water_use_mm'], 'o-', color='steelblue', label='Water Use (mm)')
        ax1.plot(usage_s.index, usage_s['rainfall_mm'],  'o-', color='seagreen',  label='Rainfall (mm)')
        ax1.set_xlabel('Month'); ax1.set_ylabel('Water (mm)')
        ax1.legend(loc='upper left')
        ax2 = ax1.twinx()
        ax2.plot(usage_s.index, usage_s['pump_hr_diesel'],    'x--', color='#e07b54', label='Pumping hr (diesel)')
        ax2.plot(usage_s.index, usage_s['pump_hr_grid_elec'], 'x--', color='#4c72b0', label='Pumping hr (electric)')
        ax2.set_ylabel('Pumping Time (hours)', color='grey')
        ax2.tick_params(axis='y', labelcolor='grey')
        ax2.legend(loc='upper right')
        plt.title(f"Monthly Water Use, Rainfall & Pumping Time\n"
                  f"{S['selected_crop']} — {S['selected_region']}, {S['selected_country']}")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout(); plt.show()

        # ── 10. Summary table ───────────────────────────────────────────────
        tbl_cols = ['pump_hr_diesel','pump_hr_grid_elec',
                    'energy_diesel_kWh','energy_grid_elec_kWh',
                    'opex_diesel_usd','opex_grid_elec_usd',
                    'ghg_diesel_kg','ghg_grid_elec_kg']
        tbl = usage_s[tbl_cols].copy()
        display_tbl = pd.concat([
            tbl.mean().to_frame('Monthly Mean').T,
            tbl.max().to_frame('Monthly Max').T,
            tbl.sum().to_frame('Annual Total').T,
        ]).round(1)
        display_tbl.columns = [
            'Pump hr\n(diesel)', 'Pump hr\n(elec)',
            'kWh\n(diesel)', 'kWh\n(elec)',
            'OPEX $\n(diesel)', 'OPEX $\n(elec)',
            'CO2 kg\n(diesel)', 'CO2 kg\n(grid)',
        ]
        nw = max(12, len(display_tbl.columns) * 1.5)
        tf, ta = plt.subplots(figsize=(nw, 2.4))
        ta.axis('off')
        tw = ta.table(cellText=display_tbl.astype(str).values,
                      colLabels=display_tbl.columns.tolist(),
                      rowLabels=display_tbl.index.tolist(),
                      cellLoc='center', loc='center')
        tw.auto_set_font_size(False); tw.set_fontsize(9); tw.scale(1.1, 1.7)
        for (r, c), cell in tw.get_celld().items():
            if r == 0:
                cell.set_text_props(weight='bold', color='white')
                cell.set_facecolor('#4c72b0')
            elif c == -1:
                cell.set_text_props(weight='bold'); cell.set_facecolor('#d9e1f2')
        tf.suptitle(f"Energy, Cost & Emissions — {S['selected_crop']} / "
                    f"{S['selected_region']}, {S['selected_country']}", fontsize=9, y=1.02)
        plt.tight_layout(); display(tf); plt.close(tf)

        # ── 11. Annual OPEX bar + GHG bar ───────────────────────────────────
        ann_opex = {ft: usage_s[f'opex_{ft}_usd'].sum() for ft in FUEL_EFFICIENCIES}
        ann_ghg  = {
            'diesel':    usage_s['ghg_diesel_kg'].sum(),
            'grid_elec': usage_s['ghg_grid_elec_kg'].sum(),
            'solar_s':   0.0,
            'solar_l':   0.0,
        }
        fl = [FUEL_LABELS[ft] for ft in FUEL_EFFICIENCIES]
        fc = [BAR_COLORS[ft]  for ft in FUEL_EFFICIENCIES]

        fig2, (ax_o, ax_g) = plt.subplots(1, 2, figsize=(13, 5))
        bs = ax_o.bar(fl, [ann_opex[ft] for ft in FUEL_EFFICIENCIES], color=fc, edgecolor='white')
        ax_o.set_title('Annual OPEX (USD)', fontweight='bold'); ax_o.set_ylabel('USD')
        for bar, v in zip(bs, ann_opex.values()):
            ax_o.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                      f'${v:,.0f}', ha='center', va='bottom', fontsize=9)
        bg = ax_g.bar(fl, [ann_ghg[ft] for ft in FUEL_EFFICIENCIES], color=fc, edgecolor='white')
        ax_g.set_title('Annual GHG Emissions', fontweight='bold'); ax_g.set_ylabel('kg CO₂')
        for bar, v in zip(bg, ann_ghg.values()):
            ax_g.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                      f'{v:,.0f}', ha='center', va='bottom', fontsize=9)
        fig2.suptitle(f"Annual OPEX & GHG — {S['selected_crop']} / "
                      f"{S['selected_region']}, {S['selected_country']}", fontweight='bold')
        plt.tight_layout(); plt.show()

        # ── 12. CapEx / OpEx / 5-yr TCO stacked bar ────────────────────────
        ann_maint = {ft: MAINTENANCE_YR[ft] for ft in FUEL_EFFICIENCIES}
        # For solar, opex = maintenance only (already set); for others add maintenance
        ann_fuel_opex = {
            'diesel':    usage_s['opex_diesel_usd'].sum() + ann_maint['diesel'],
            'grid_elec': usage_s['opex_grid_elec_usd'].sum() + ann_maint['grid_elec'],
            'solar_s':   ann_maint['solar_s'],
            'solar_l':   ann_maint['solar_l'],
        }
        capex_vals   = [CAPEX[ft] for ft in FUEL_EFFICIENCIES]
        tco5_opex    = [ann_fuel_opex[ft] * TCO_YEARS for ft in FUEL_EFFICIENCIES]
        tco5_total   = [c + o for c, o in zip(capex_vals, tco5_opex)]

        x = np.arange(len(fl))
        w = 0.5
        fig3, ax3 = plt.subplots(figsize=(10, 6))
        b1 = ax3.bar(x, capex_vals,  w, label='CapEx (one-time)', color=fc, alpha=0.9, edgecolor='white')
        b2 = ax3.bar(x, tco5_opex,   w, bottom=capex_vals,
                     label=f'{TCO_YEARS}-yr OpEx + Maint.', color=fc, alpha=0.45, edgecolor='white', hatch='//')
        for xi, (cv, ov, tot) in enumerate(zip(capex_vals, tco5_opex, tco5_total)):
            ax3.text(xi, tot + 15, f'${tot:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax3.set_xticks(x); ax3.set_xticklabels(fl)
        ax3.set_ylabel('USD'); ax3.set_title(f'{TCO_YEARS}-Year Total Cost of Ownership\n'
                                              f'CapEx (solid) + Cumulative OpEx & Maintenance (hatched)',
                                              fontweight='bold')
        ax3.legend(); plt.tight_layout(); plt.show()

        # ── 13. Breakeven / cumulative cost chart ───────────────────────────
        months_x = np.arange(0, TCO_YEARS * 12 + 1)
        fig4, ax4 = plt.subplots(figsize=(11, 5))
        for ft in FUEL_EFFICIENCIES:
            monthly_total = ann_fuel_opex[ft] / 12
            cumulative    = CAPEX[ft] + monthly_total * months_x
            ax4.plot(months_x, cumulative, color=BAR_COLORS[ft],
                     linewidth=2.2, label=FUEL_LABELS[ft])
        ax4.set_xlabel('Month'); ax4.set_ylabel('Cumulative Cost (USD)')
        ax4.set_title(f'Cumulative Cost Over {TCO_YEARS} Years\n'
                      f'(CapEx + running OpEx & maintenance)', fontweight='bold')
        ax4.xaxis.set_major_locator(plt.MultipleLocator(6))
        ax4.grid(True, linestyle='--', alpha=0.5); ax4.legend(); plt.tight_layout(); plt.show()

        # ── 14. Power capacity check ────────────────────────────────────────
        if power_errors:
            err_html = "<h3 style='color:#d9534f'>⚠️ Power Capacity Warnings</h3><ul>"
            for e in power_errors:
                err_html += f"<li>{e}</li>"
            err_html += "</ul>"
            display(widgets.HTML(err_html))
        else:
            display(widgets.HTML("<h3 style='color:#5cb85c'>✅ All pump capacities sufficient</h3>"))


# ── Wire up & display ──────────────────────────────────────────────────────
btn_load_crop.on_click(on_load_crop)
btn_select_country.on_click(on_select_country)
btn_select_region.on_click(on_select_region)
btn_set_pump.on_click(on_set_pump)
btn_run.on_click(on_run)

display(widgets.VBox([
    widgets.HTML('<h3>Step 1 — Select Crop</h3>'),
    crop_selector, btn_load_crop, out_crop,
    step2_box, step3_box, step4_box, step5_box,
]))
